# ReceiptVLM — score a saved adapter (no training)Loads an already-trained LoRA adapter and runs the accuracy gate against the WildReceipttest split, using the project's own `eval.py`. Nothing here trains, so it is cheap(~15 min) and safe to re-run.**Set `LOAD_4BIT` below to pick the serving precision.** That switch is the whole point ofthis notebook: an adapter only works against the base it was fit to, so the question"can this NF4-trained adapter be served in fp16?" is answered by running it both ways andcomparing.It matters for deployment because HF ZeroGPU — the free GPU host for the public demo —does not reliably support bitsandbytes: reports include bitsandbytes being compiledwithout GPU support there, and the usual workaround (loading inside the `@spaces.GPU`function) contradicts ZeroGPU's requirement to load at module scope. If the adapter holdsup in fp16, bitsandbytes never enters the picture.## Inputs to attach| input | how ||---|---|| the adapter (`adapter_config.json` + `adapter_model.safetensors`) | **+ Add Input → Notebook Output** of the training run, or upload as a Dataset || `test.jsonl`, `finetuned_test.jsonl` | the `receiptvlm-data` Dataset || `eval.py`, `repair.py`, `schema.py`, `zeroshot.py` | the same Dataset (all four in one folder) || WildReceipt images | attached as a Dataset, or downloaded below |Use **GPU T4 x2** and **Internet on**.

In [ ]:
# Pin torch to whatever Kaggle preinstalled, so resolving the packages below cannot pull a# wheel built for a narrower set of GPU architectures.import torchopen("constraints.txt", "w").write(f"torch=={torch.__version__.split('+')[0]}\n")

In [ ]:
!pip install -q -U -c constraints.txt "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes>=0.44" "safetensors>=0.4"

## Configuration

In [ ]:
import json, sys, timefrom pathlib import Pathimport torch# ---- the switch this notebook exists for -----------------------------------------LOAD_4BIT = False        # False = fp16 serving; True = NF4, matching QLoRA training# ----------------------------------------------------------------------------------BASE_MODEL   = "Qwen/Qwen2.5-VL-3B-Instruct"IMAGE_RESIZE = (768, 1024)      # must match train.py's --image-resizeMAX_NEW_TOKENS = 1536SCORE_LIMIT  = 30               # same receipts scripts/validate_peft_adapter.py usesEXPECT_MODULES = 252            # the MLX run's adapted-module countSCHEMA_KEYS = ["store", "date", "tax", "tip", "subtotal", "total", "line_items"]PROMPT = ("Extract the receipt fields as JSON with keys store, date, tax, tip, "          "subtotal, total, line_items (each {name, price}). Use null for missing "          "scalar fields and [] for no line items.")OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("out")OUT_ROOT.mkdir(parents=True, exist_ok=True)if not torch.cuda.is_available():    raise SystemExit("No GPU. Set Accelerator -> GPU T4 x2.")major, minor = torch.cuda.get_device_capability(0)sm = f"sm_{major}{minor}"if sm not in torch.cuda.get_arch_list():    raise SystemExit(        f"This torch has no kernels for {torch.cuda.get_device_name(0)} ({sm}); "        f"it targets {torch.cuda.get_arch_list()}. Switch to 'GPU T4 x2'."    )if LOAD_4BIT and (major, minor) < (7, 5):    raise SystemExit(f"bitsandbytes 4-bit needs sm_75+; this is {sm}.")print(f"{torch.cuda.get_device_name(0)} ({sm}) | serving in "      f"{'NF4' if LOAD_4BIT else 'fp16'}")

## Locate inputs

In [ ]:
INPUT_ROOT = Path("/kaggle/input")def find_input(name, must_contain=None):    '''Find a file/dir anywhere under /kaggle/input, then the local repo.    Searched at any depth because Kaggle preserves upload structure: the same file lands    at <dataset>/test.jsonl if uploaded flat but <dataset>/data/processed/test.jsonl if    the repo folders were kept.    '''    def ok(p):        return (p / must_contain).exists() if must_contain else p.exists()    if INPUT_ROOT.exists():        for p in sorted(INPUT_ROOT.rglob(name)):            if ok(p):                return p    for p in (Path("data/processed") / name, Path("src") / name,              Path("checkpoints") / name, Path(name)):        if p.exists() and ok(p):            return p    return None# The training run's output holds several adapters -- the periodic checkpoints# (checkpoints/step_2000, step_2250) as well as final_peft. A plain sorted() pick lands on# "checkpoints/..." before "final_peft/..." alphabetically, which would silently score a# mid-training checkpoint. Prefer final_peft, and say which one was chosen.ADAPTER_DIR_OVERRIDE = None      # set to a path to score a specific checkpoint_adapter_cfgs = ([Path(ADAPTER_DIR_OVERRIDE) / "adapter_config.json"]                 if ADAPTER_DIR_OVERRIDE else                 sorted(INPUT_ROOT.rglob("adapter_config.json"))                 if INPUT_ROOT.exists() else [])if not _adapter_cfgs:    local = Path("checkpoints/final_peft/adapter_config.json")    _adapter_cfgs = [local] if local.exists() else []adapter_cfg = next((p for p in _adapter_cfgs if "final_peft" in str(p)),                   _adapter_cfgs[0] if _adapter_cfgs else None)adapter_dir = adapter_cfg.parent if adapter_cfg else Noneif len(_adapter_cfgs) > 1:    print(f"found {len(_adapter_cfgs)} adapters; using {adapter_dir}")    for p in _adapter_cfgs:        print("   ", "->" if p == adapter_cfg else "  ", p.parent)test_jsonl = find_input("test.jsonl")reference = find_input("finetuned_test.jsonl")eval_py = find_input("eval.py")img_root = find_input("wildreceipt", must_contain="image_files")if img_root is None:    dest = OUT_ROOT / "wildreceipt"    dest.mkdir(parents=True, exist_ok=True)    print("downloading WildReceipt (~179 MB)...")    !curl -sL -o {OUT_ROOT}/wildreceipt.tar https://download.openmmlab.com/mmocr/data/wildreceipt.tar    !tar -xf {OUT_ROOT}/wildreceipt.tar -C {dest} --strip-components=1    img_root = destSIBLINGS = ["repair.py", "schema.py", "zeroshot.py"]missing = [s for s in SIBLINGS if eval_py and not (eval_py.parent / s).exists()]print("adapter   :", adapter_dir)print("test.jsonl:", test_jsonl)print("reference :", reference)print("eval.py   :", eval_py, f"(missing siblings: {missing or 'none'})")print("images    :", img_root)if adapter_dir is None:    raise SystemExit(        "No adapter found. Attach the training run's output via "        "'+ Add Input -> Notebook Output', or upload final_peft/ as a Dataset.\n"        f"Attached files: {sorted(str(p.relative_to(INPUT_ROOT)) for p in INPUT_ROOT.rglob('*') if p.is_file())[:25]}"    )if eval_py is None or test_jsonl is None or missing:    raise SystemExit(        "Scoring inputs incomplete. Need test.jsonl plus eval.py, repair.py, schema.py "        "and zeroshot.py (the four .py files in one folder)."    )cfg = json.loads(adapter_cfg.read_text())print(f"\nadapter: r={cfg['r']} lora_alpha={cfg['lora_alpha']} "      f"-> scale {cfg['lora_alpha'] / cfg['r']}")assert cfg["lora_alpha"] / cfg["r"] == 0.125, "scale differs from the MLX run's 0.125"

## Load the model

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGenerationfrom peft import PeftModelkwargs = {"dtype": torch.float16, "attn_implementation": "sdpa"}if LOAD_4BIT:    from transformers import BitsAndBytesConfig    kwargs["quantization_config"] = BitsAndBytesConfig(        load_in_4bit=True, bnb_4bit_quant_type="nf4",        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)base = Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, **kwargs)processor = AutoProcessor.from_pretrained(BASE_MODEL)model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)# bitsandbytes has already placed the quantized weights; .to() on a 4-bit model raises.model = (model.eval() if LOAD_4BIT else model.to("cuda").eval())model.config.use_cache = Truen_inj = sum(1 for _, m in model.named_modules()            if hasattr(getattr(m, "lora_A", None), "keys"))n_loaded = sum(1 for _, m in model.named_modules()               if hasattr(getattr(m, "lora_A", None), "keys")               and any(float(p.detach().abs().max()) > 0 for p in m.lora_B.parameters()))print(f"lora modules injected: {n_inj}   carrying trained weights: {n_loaded}")# peft zero-initializes lora_B, so an all-zero B means the weights never loaded -- the# silent no-op that makes a base model masquerade as a fine-tuned one.assert n_inj == EXPECT_MODULES, f"expected {EXPECT_MODULES} modules, got {n_inj}"assert n_loaded == n_inj, "some modules have a zero lora_B: weights did not load"print("adapter loaded")

## Inference helper

In [ ]:
from PIL import Imagedef fit_within(img, max_w, max_h):    '''Port of mlx_vlm.utils.resize_image: fit the box, aspect preserved, no clamp.'''    ratio = min(max_w / img.width, max_h / img.height)    return img.resize((int(img.width * ratio), int(img.height * ratio)))def generate_raw(image_path, max_new_tokens=MAX_NEW_TOKENS):    img = fit_within(Image.open(image_path).convert("RGB"), *IMAGE_RESIZE)    messages = [{"role": "user", "content": [{"type": "image"},                                             {"type": "text", "text": PROMPT}]}]    text = processor.apply_chat_template(messages, tokenize=False,                                         add_generation_prompt=True)    enc = processor(text=[text], images=[img], return_tensors="pt").to("cuda")    with torch.inference_mode():        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)    return processor.batch_decode(out[:, enc["input_ids"].shape[1]:],                                   skip_special_tokens=True)[0]print("ready")

## The accuracy gateScores the first `SCORE_LIMIT` receipts of `test.jsonl` in file order — the same subset`scripts/validate_peft_adapter.py` uses, so results are directly comparable across runs.Reference points on these 30 receipts: MLX run **0.785**, NF4-served retrain **0.722**,fp16-transferred original adapter **0.288**.

In [ ]:
sys.path.insert(0, str(eval_py.parent))from eval import bootstrap_micro_f1, evaluate, paired_bootstrap_testfrom repair import repair_jsonfrom zeroshot import normalize          # mlx import is guarded, so this is safe heregold_all = {json.loads(l)["image_id"]: json.loads(l)            for l in test_jsonl.open() if l.strip()}score_ids = list(gold_all)[:SCORE_LIMIT]absent = [i for i in score_ids if not (img_root / i).exists()]assert not absent, f"{len(absent)} test images missing, e.g. {absent[0]}"preds, statuses = {}, {}t0 = time.time()for n, iid in enumerate(score_ids, 1):    raw = generate_raw(img_root / iid)    parsed, status = repair_json(raw)    statuses[status] = statuses.get(status, 0) + 1    preds[iid] = {"image_id": iid, **normalize(parsed)}    print(f"  [{n}/{len(score_ids)}] {status:<22} {iid.split('/')[-1][:26]}", flush=True)elapsed = time.time() - t0print(f"\nscored {len(preds)} receipts in {elapsed:.0f}s "      f"({elapsed / len(preds):.1f}s each) | repair: {statuses}")gold = {i: gold_all[i] for i in preds}per_field, micro, n_scored = evaluate(gold, preds)lo, hi = bootstrap_micro_f1(gold, preds, n=1000)label = f"{'NF4' if LOAD_4BIT else 'fp16'} serving"print(f"\n{label:<16} micro-F1 {micro[2]:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (n={n_scored})")for field, (p, r, f1) in sorted(per_field.items()):    print(f"  {field:<18} P {p:.3f}  R {r:.3f}  F1 {f1:.3f}")if reference is not None:    ref_all = {json.loads(l)["image_id"]: json.loads(l)               for l in reference.open() if l.strip()}    shared = [i for i in preds if i in ref_all]    if shared:        g = {i: gold_all[i] for i in shared}        a = {i: ref_all[i] for i in shared}      # MLX run        b = {i: preds[i] for i in shared}        # this adapter        _, ref_micro, _ = evaluate(g, a)        _, own_micro, _ = evaluate(g, b)        res = paired_bootstrap_test(g, a, b, n=1000)   # reports b - a        print(f"\nPaired against {reference.name} on {len(shared)} receipts:")        print(f"  MLX reference  micro-F1 {ref_micro[2]:.3f}")        print(f"  {label:<13}  micro-F1 {own_micro[2]:.3f}")        print(f"  delta {res['mean_diff']:+.3f}  "              f"95% CI [{res['ci'][0]:+.3f}, {res['ci'][1]:+.3f}]  p={res['p_approx']:.3f}")        regressed = res["mean_diff"] < 0 and res["p_approx"] < 0.05        print("\n" + ("GATE FAIL: significantly behind the MLX run."                      if regressed else                      "GATE PASS: no significant regression vs. the MLX run."))tag = "nf4" if LOAD_4BIT else "fp16"out = OUT_ROOT / f"{tag}_test_predictions.jsonl"out.write_text("".join(json.dumps(v) + "\n" for v in preds.values()))print("\nwrote", out)

## Reading the result- **fp16 within noise of 0.722** → serve fp16 on ZeroGPU and skip bitsandbytes entirely.  Leave `RECEIPTVLM_LOAD_4BIT` unset.- **fp16 collapses toward 0.288** → the adapter is bound to its NF4 base, so the Space  must run NF4 (`RECEIPTVLM_LOAD_4BIT=1`). Given ZeroGPU's bitsandbytes problems, that  likely means a different host.Either way the adapter itself is unchanged — this only decides where it can run.